# Lab01：概率 BC 与多峰动作

比较 MSE、单 Gaussian 与 mixture：低 mean MSE 是否代表能采到合理模式？

**运行方式**：上传本 `.ipynb` 到 Google Colab，连接 Python runtime，从上到下运行。
默认 CPU 即可；本 notebook 自带代码和合成数据，不需要克隆仓库、上传论文、挂载 Drive 或配置本地环境。
所有输出留空，首次运行后才产生实际结果；不要把材料准备状态当作已复现实验。

如果所选云端 runtime 缺少包，可在 Colab 新单元执行 `%pip install numpy matplotlib pandas scikit-learn torch`，
然后重启 runtime。不要为了本课程在本机安装依赖。


## 1. 环境与随机性
记录版本和 seed。换 seed 时从本单元重新运行；图中用英文标签避免依赖额外中文字体。


In [ ]:
import json
import math
import platform
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
print('Python:', platform.python_version(), '| NumPy:', np.__version__, '| seed:', SEED)
import torch
from torch import nn
import torch.nn.functional as F
torch.manual_seed(SEED)
torch.set_num_threads(2)
print('torch:', torch.__version__, '| teaching device: CPU')


## 2. 自包含模型与数据定义
以下定义迁移自原 `01_probabilistic_bc.py`，不读取该脚本。隐藏 branch 不输入 policy，split 按 episode。


In [ ]:
from __future__ import annotations
import json
import math
from pathlib import Path
import torch
from torch import nn


### 定义：`Policy`
先看输入输出和 shape，再运行定义。本单元不启动训练。


In [ ]:
class Policy(nn.Module):
    def __init__(self, kind: str) -> None:
        super().__init__()
        self.kind = kind
        output_dim = {"mse": 1, "gaussian": 2, "mixture": 6}[kind]
        self.body = nn.Sequential(
            nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh()
        )
        self.head = nn.Linear(32, output_dim)
        if kind == "mixture":
            # Break component symmetry; neither true branch is supplied as input.
            with torch.no_grad():
                self.head.weight.mul_(0.1)
                self.head.bias.copy_(torch.tensor([0.0, 0.0, -1.0, 1.0, -1.5, -1.5]))

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        return self.head(self.body(obs))


### 定义：`log_normal`
先看输入输出和 shape，再运行定义。本单元不启动训练。


In [ ]:
def log_normal(action: torch.Tensor, mean: torch.Tensor, log_std: torch.Tensor) -> torch.Tensor:
    return -0.5 * ((action - mean) / log_std.exp()).square() - log_std - 0.5 * math.log(2 * math.pi)


### 定义：`loss`
先看输入输出和 shape，再运行定义。本单元不启动训练。


In [ ]:
def loss(policy: Policy, obs: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
    output = policy(obs)
    if policy.kind == "mse":
        return (output - action).square().mean()
    if policy.kind == "gaussian":
        mean, log_std = output[:, :1], output[:, 1:2].clamp(-5.0, 2.0)
        return -log_normal(action, mean, log_std).mean()
    log_weights = output[:, :2].log_softmax(dim=-1)
    means = output[:, 2:4]
    log_stds = output[:, 4:6].clamp(-5.0, 2.0)
    component_log_probs = log_weights + log_normal(action, means, log_stds)
    return -torch.logsumexp(component_log_probs, dim=-1).mean()


### 定义：`make_data`
先看输入输出和 shape，再运行定义。本单元不启动训练。


In [ ]:
def make_data(seed: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, dict]:
    generator = torch.Generator().manual_seed(seed)
    n_episodes, episode_length = 64, 64
    obs = torch.linspace(-1.0, 1.0, episode_length).reshape(1, episode_length, 1)
    obs = obs.repeat(n_episodes, 1, 1)
    obs = obs + 0.03 * torch.randn(obs.shape, generator=generator)
    branches = (2 * (torch.arange(n_episodes) % 2) - 1).reshape(n_episodes, 1, 1)
    action = 0.5 * obs + branches + 0.05 * torch.randn(obs.shape, generator=generator)

    # Split balanced episode pairs; no episode appears in both partitions.
    pairs = torch.randperm(n_episodes // 2, generator=generator)
    train_ids = torch.stack((2 * pairs[:24], 2 * pairs[:24] + 1), dim=1).flatten()
    validation_ids = torch.stack((2 * pairs[24:], 2 * pairs[24:] + 1), dim=1).flatten()
    metadata = {
        "n_episodes": n_episodes,
        "episode_length": episode_length,
        "train_episode_ids": train_ids.tolist(),
        "validation_episode_ids": validation_ids.tolist(),
        "true_noise_std": 0.05,
        "observation": "scalar o; branch b is hidden from this single-step policy",
        "expert_action": "a = 0.5 * o + b + Normal(0, 0.05^2), b in {-1, +1}",
        "limitation": "Synthetic episode grouping only; no transition dynamics or robot rollout.",
    }
    return (
        obs[train_ids].reshape(-1, 1),
        action[train_ids].reshape(-1, 1),
        obs[validation_ids].reshape(-1, 1),
        action[validation_ids].reshape(-1, 1),
        metadata,
    )


### 定义：`mean_prediction`
先看输入输出和 shape，再运行定义。本单元不启动训练。


In [ ]:
@torch.no_grad()
def mean_prediction(policy: Policy, obs: torch.Tensor) -> torch.Tensor:
    output = policy(obs)
    if policy.kind == "mse":
        return output
    if policy.kind == "gaussian":
        return output[:, :1]
    return (output[:, :2].softmax(dim=-1) * output[:, 2:4]).sum(dim=-1, keepdim=True)


### 定义：`sample_actions`
先看输入输出和 shape，再运行定义。本单元不启动训练。


In [ ]:
@torch.no_grad()
def sample_actions(policy: Policy, obs: torch.Tensor) -> torch.Tensor:
    output = policy(obs)
    if policy.kind == "mse":
        return output
    if policy.kind == "gaussian":
        return output[:, :1] + output[:, 1:2].clamp(-5.0, 2.0).exp() * torch.randn_like(obs)
    components = torch.multinomial(output[:, :2].softmax(dim=-1), num_samples=1)
    means = output[:, 2:4].gather(1, components)
    stds = output[:, 4:6].clamp(-5.0, 2.0).exp().gather(1, components)
    return means + stds * torch.randn_like(obs)


### 定义：`evaluate`
先看输入输出和 shape，再运行定义。本单元不启动训练。


In [ ]:
@torch.no_grad()
def evaluate(policy: Policy, obs: torch.Tensor, action: torch.Tensor) -> dict:
    policy.eval()
    mean = mean_prediction(policy, obs)
    sample_obs = 2 * torch.rand(8192, 1) - 1
    samples = sample_actions(policy, sample_obs)
    center = 0.5 * sample_obs
    distance = torch.minimum((samples - (center - 1)).abs(), (samples - (center + 1)).abs())
    output_zero = policy(torch.zeros(1, 1))
    if policy.kind == "mse":
        at_zero = {"mean": output_zero.item()}
    elif policy.kind == "gaussian":
        at_zero = {
            "mean": output_zero[0, 0].item(),
            "std": output_zero[0, 1].clamp(-5.0, 2.0).exp().item(),
        }
    else:
        at_zero = {
            "weights": output_zero[0, :2].softmax(dim=-1).tolist(),
            "means": output_zero[0, 2:4].tolist(),
            "stds": output_zero[0, 4:6].clamp(-5.0, 2.0).exp().tolist(),
        }
    return {
        "held_out_mean_prediction_mse": (mean - action).square().mean().item(),
        "held_out_nll": None if policy.kind == "mse" else loss(policy, obs, action).item(),
        "sampled_mode_compatibility": (distance < 0.15).float().mean().item(),
        "mode_compatibility_threshold": 0.15,
        "sampled_positive_branch_fraction": (samples > center).float().mean().item(),
        "distribution_at_o_zero": at_zero,
    }


## 3. 配置、训练与 held-out 评价
三模型使用同一数据与 batch/steps。NLL 只在概率模型间有定义；mode compatibility 是合成模式距离，不是机器人成功率。


In [ ]:
STEPS = 600
BATCH_SIZE = 128
train_obs, train_action, val_obs, val_action, metadata = make_data(SEED)
assert set(metadata['train_episode_ids']).isdisjoint(metadata['validation_episode_ids'])
policies, traces, metrics = {}, {}, {}
for index, kind in enumerate(('mse', 'gaussian', 'mixture')):
    torch.manual_seed(SEED + 100 * index)
    policy = Policy(kind)
    optimizer = torch.optim.Adam(policy.parameters(), lr=0.003)
    history = []
    policy.train()
    for step in range(STEPS):
        indices = torch.randint(len(train_obs), (BATCH_SIZE,))
        objective = loss(policy, train_obs[indices], train_action[indices])
        optimizer.zero_grad(set_to_none=True)
        objective.backward()
        optimizer.step()
        if step % 10 == 0:
            history.append((step, float(objective.detach())))
    policies[kind], traces[kind] = policy, history
    metrics[kind] = evaluate(policy, val_obs, val_action)
results = {'seed': SEED, 'steps': STEPS, 'data': metadata, 'models': metrics,
           'scope': 'Synthetic conditional distribution; no robot rollout; validation split only.'}
display(pd.DataFrame(metrics).T.drop(columns='distribution_at_o_zero'))


## 4. 观察 o=0 的采样分布
模型的 mean 与 sample 分开看。两条 expert modes 在 -1/+1；单 Gaussian 的均值可能仍在中间。


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharex=True, sharey=True)
with torch.no_grad():
    for axis, (kind, policy) in zip(axes, policies.items()):
        samples = sample_actions(policy, torch.zeros(4000, 1)).numpy().ravel()
        axis.hist(samples, bins=60, range=(-2, 2), density=True, alpha=0.7)
        axis.axvline(-1, color='black', linestyle='--')
        axis.axvline(1, color='black', linestyle='--')
        axis.set(title=kind, xlabel='action at o=0')
axes[0].set_ylabel('density')
plt.tight_layout(); plt.show()


## 面试追问与实验记录
1. 推导固定方差 Gaussian NLL 与 MSE 的关系。
2. 为什么 mixture 的 mean 仍可能在两个 modes 之间？
3. 固定数据，仅减少 steps；解释优化不充分与分布假设错误的区别。

记录 Question / Data / Config / Metric / Observation / Explanation / Limitation。
改变实验设置时不要反复用 test 选择答案；保存 notebook 后再下载 `.ipynb` 保留输出。


## 可选：导出指标
默认关闭。需要时打开开关；文件只写到当前 **Colab runtime** 的 `outputs/`。runtime 临时文件不等于已永久保存，可使用 Colab 文件面板下载。


In [ ]:
EXPORT_RESULTS = False
if EXPORT_RESULTS:
    destination = Path('outputs')
    destination.mkdir(exist_ok=True)
    (destination / 'metrics.json').write_text(
        json.dumps(results, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    print('Saved outputs/metrics.json in the Colab runtime; download it from the Files panel.')
